# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, examine, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing dataset entities by their `@id` fields as per the Croissant schema.

### Dataset Source
The dataset Croissant schema is available at:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
We load the dataset's metadata and records using `mlcroissant`. This includes printing the dataset's main description for context.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Let's examine the record sets, their fields, and how to interact with them. All references will use the `@id` value of each entity, per Croissant best practices.

In [ ]:
from pprint import pprint

record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")

for record_set in record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id}; Name: {field.name}; Data type: {field.data_type}")
    print()

## 3. Data Extraction
We now extract data from a specific record set into a DataFrame. Use `@id` fields to specify which record set and which fields you want to load/inspect. Below, we inspect all available record sets and extract the one containing the main experimental data.

In [ ]:
# Prepare a dictionary of DataFrames for each record set
dataframes = {}
all_record_set_ids = [rs.id for rs in metadata.record_sets]
print("All RecordSet @id values:", all_record_set_ids)

# Choose the first record set for demonstration (replace as needed)
selected_record_set_id = all_record_set_ids[0] if all_record_set_ids else None
if selected_record_set_id is None:
    raise RuntimeError("No record sets defined in the schema.")

# Load records from each record set
for rs_id in all_record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

print(f"Columns in record set '{selected_record_set_id}':\n", dataframes[selected_record_set_id].columns.tolist())
dataframes[selected_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main record set. We'll pick a numeric field (by its `@id`), filter records, normalize the data, and optionally group by another field. (Adjust the `numeric_field_id` and `group_field_id` based on overview above and your objectives.)

In [ ]:
# Identify a numeric field (adjust as required based on dataset schema)
# Let's enumerate columns again to decide interactively
df = dataframes[selected_record_set_id]
print("Available columns in selected record set:")
print(df.columns.tolist())

# Example: Suppose 'cr:Abundance' or 'cr:MW' exists as a field @id.
numeric_field_id = None
for col in df.columns:
    # Heuristics: find first numeric-typed field
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # Fallback to user-provided string
    numeric_field_id = df.columns[0]  # may need adjustment
print(f"Using '{numeric_field_id}' as the numeric field for EDA.\n")

# Filtering records where value > threshold
threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold} (mean value):")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a categorical field if present
group_field_id = None
for col in df.columns:
    if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
        group_field_id = col
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped data by '{group_field_id}':")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and the effect of normalization. Adjust field choices as needed for your analytical objectives.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")

plt.subplot(1, 2, 2)
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
plt.title(f"Normalized {numeric_field_id} (filtered)")
plt.tight_layout()
plt.show()

if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
- We demonstrated loading and exploring a dataset described by a Croissant schema with `mlcroissant`.
- We used `@id` fields to refer to record sets and fields for full provenance and reproducibility.
- We extracted and processed records, normalized and grouped data, and visualized distributions.

This approach ensures transparent and reproducible FAIR data science workflows, leveraging the power of structured metadata and modern data standards.